---
## TUGAS MANDIRI (Dikerjakan Selama 1 Minggu)

> **Tenggat waktu:** dikumpulkan paling lambat **sebelum Pertemuan 13 dimulai**.
> **Sifat tugas:** individu.

### Konteks / Skenario

"Kopi Nusantara" ingin membangun model untuk memprediksi apakah seorang pelanggan akan memberikan **ulasan positif (rating ≥ 4) atau tidak**, berdasarkan atribut pesanan mereka — berguna untuk mengidentifikasi kombinasi menu/cabang yang berisiko mengecewakan pelanggan sebelum benar-benar terjadi.

### Menyiapkan Dataset

Jalankan cell berikut untuk membuat dataset.

In [7]:
import numpy as np
import pandas as pd

np.random.seed(212)
n = 3000
menu_list = ["Kopi Susu", "Americano", "Cappuccino", "Roti Bakar", "Kentang Goreng", "Es Teh"]
cabang_list = ["Magelang", "Yogyakarta", "Semarang"]
waktu_list = ["Pagi", "Siang", "Sore", "Malam"]

data_tugas12 = pd.DataFrame({
    "nama_menu": np.random.choice(menu_list, size=n),
    "cabang": np.random.choice(cabang_list, size=n),
    "waktu_pesan": np.random.choice(waktu_list, size=n),
    "qty": np.random.randint(1, 6, size=n),
    "waktu_tunggu_menit": np.random.randint(2, 25, size=n),
})

# Simulasi: waktu tunggu lebih lama cenderung (namun tidak selalu) menurunkan peluang ulasan positif
prob_positif = 1 / (1 + np.exp((data_tugas12["waktu_tunggu_menit"] - 12) / 4))
data_tugas12["ulasan_positif"] = (np.random.rand(n) < prob_positif).astype(int)

data_tugas12.to_csv("tugas12_ulasan.csv", index=False)
print(f"Dataset berhasil dibuat: {len(data_tugas12)} baris")
print("Distribusi label:")
print(data_tugas12["ulasan_positif"].value_counts())

Dataset berhasil dibuat: 3000 baris
Distribusi label:
ulasan_positif
0    1610
1    1390
Name: count, dtype: int64


### Instruksi Pengerjaan

Buat notebook baru **`Tugas12_[NPM]_[Nama Lengkap].ipynb`**, buat `SparkSession`, baca `tugas12_ulasan.csv`, lalu kerjakan bagian **A sampai E** berikut:

---

**A. Membangun ML Pipeline** *(bobot 25%)*

Buat `Pipeline` dengan tahapan: `StringIndexer` untuk `nama_menu`, `cabang`, `waktu_pesan` → `OneHotEncoder` untuk ketiganya → `VectorAssembler` (gabungkan hasil one-hot encoding dengan `qty` dan `waktu_tunggu_menit`) → `LogisticRegression` dengan `labelCol="ulasan_positif"`.

**B. Pelatihan & Evaluasi** *(bobot 25%)*

Bagi data 80:20 (`seed=77`), latih pipeline, lalu evaluasi pada data testing menggunakan **accuracy**, **F1-score**, dan **AUC-ROC**. Tampilkan juga *confusion matrix* sederhana menggunakan `groupBy`.

**C. Model Pembanding** *(bobot 20%)*

Latih **`RandomForestClassifier`** (`numTrees=30`) menggunakan pipeline dengan lima tahap feature engineering yang identik dari bagian A (hanya tahap terakhir yang diganti). Bandingkan ketiga metrik evaluasinya dengan model Logistic Regression pada bagian B.

**D. Analisis Kesalahan Model** *(bobot 15%)*

Dari hasil prediksi model **terbaik** (pilih salah satu dari B atau C berdasarkan AUC tertinggi), tampilkan 10 baris di mana model **salah** memprediksi (`label != prediction`). Amati: apakah ada pola tertentu pada kesalahan tersebut (mis. mayoritas terjadi pada `waktu_tunggu_menit` yang berada di sekitar ambang batas tertentu)?

**E. Kesimpulan Bisnis** *(bobot 15%)*

Tulis pada markdown cell (**minimal 100 kata**): berdasarkan model yang telah dibangun, apa rekomendasi konkret yang dapat diberikan kepada manajemen "Kopi Nusantara" untuk meningkatkan kepuasan pelanggan? Kaitkan dengan fitur yang paling berpengaruh terhadap prediksi (boleh merujuk pada pola yang ditemukan di bagian D).

---

In [8]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier
from pyspark.ml import Pipeline
from pyspark.ml.evaluation import MulticlassClassificationEvaluator, BinaryClassificationEvaluator

spark = SparkSession.builder \
    .appName("Tugas12-KopiNusantara") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
spark.conf.set("spark.sql.shuffle.partitions", "4")

df = spark.read.csv("tugas12_ulasan.csv", header=True, inferSchema=True)
df.show(5)

+--------------+----------+-----------+---+------------------+--------------+
|     nama_menu|    cabang|waktu_pesan|qty|waktu_tunggu_menit|ulasan_positif|
+--------------+----------+-----------+---+------------------+--------------+
|     Kopi Susu|Yogyakarta|       Sore|  5|                 5|             1|
|        Es Teh|Yogyakarta|       Pagi|  1|                21|             0|
|    Cappuccino|Yogyakarta|       Sore|  5|                15|             0|
|Kentang Goreng|  Semarang|      Malam|  2|                 5|             1|
|     Americano|Yogyakarta|      Malam|  2|                10|             0|
+--------------+----------+-----------+---+------------------+--------------+
only showing top 5 rows



In [9]:
# A. Membangun ML Pipeline 
idx_menu = StringIndexer(inputCol="nama_menu", outputCol="menu_idx")
idx_cabang = StringIndexer(inputCol="cabang", outputCol="cabang_idx")
idx_waktu = StringIndexer(inputCol="waktu_pesan", outputCol="waktu_idx")

# OneHotEncoder
ohe = OneHotEncoder(
    inputCols=["menu_idx", "cabang_idx", "waktu_idx"],
    outputCols=["menu_vec", "cabang_vec", "waktu_vec"]
)

# VectorAssembler
assembler = VectorAssembler(
    inputCols=["menu_vec", "cabang_vec", "waktu_vec", "qty", "waktu_tunggu_menit"],
    outputCol="features"
)

# Logistic Regression
lr = LogisticRegression(featuresCol="features", labelCol="ulasan_positif")

pipeline_lr = Pipeline(stages=[idx_menu, idx_cabang, idx_waktu, ohe, assembler, lr])
print("Pipeline Logistic Regression berhasil didefinisikan.")

Pipeline Logistic Regression berhasil didefinisikan.


In [10]:
# B. Pelatihan & Evaluasi 
train, test = df.randomSplit([0.8, 0.2], seed=77)
model_lr = pipeline_lr.fit(train)
pred_lr = model_lr.transform(test)

ev_acc = MulticlassClassificationEvaluator(labelCol="ulasan_positif", predictionCol="prediction", metricName="accuracy")
ev_f1 = MulticlassClassificationEvaluator(labelCol="ulasan_positif", predictionCol="prediction", metricName="f1")
ev_auc = BinaryClassificationEvaluator(labelCol="ulasan_positif", metricName="areaUnderROC")

acc_lr = ev_acc.evaluate(pred_lr)
f1_lr = ev_f1.evaluate(pred_lr)
auc_lr = ev_auc.evaluate(pred_lr)

print("=== Logistic Regression ===")
print(f"Accuracy : {acc_lr:.4f}")
print(f"F1-score : {f1_lr:.4f}")
print(f"AUC-ROC  : {auc_lr:.4f}")

print("\nConfusion Matrix:")
pred_lr.groupBy("ulasan_positif", "prediction").count().orderBy("ulasan_positif", "prediction").show()

=== Logistic Regression ===
Accuracy : 0.7929
F1-score : 0.7926
AUC-ROC  : 0.8650

Confusion Matrix:
+--------------+----------+-----+
|ulasan_positif|prediction|count|
+--------------+----------+-----+
|             0|       0.0|  257|
|             0|       1.0|   57|
|             1|       0.0|   65|
|             1|       1.0|  210|
+--------------+----------+-----+



In [11]:
# C. Model Pembanding
rf = RandomForestClassifier(featuresCol="features", labelCol="ulasan_positif", numTrees=30)
pipeline_rf = Pipeline(stages=[idx_menu, idx_cabang, idx_waktu, ohe, assembler, rf])

model_rf = pipeline_rf.fit(train)
pred_rf = model_rf.transform(test)

acc_rf = ev_acc.evaluate(pred_rf)
f1_rf = ev_f1.evaluate(pred_rf)
auc_rf = ev_auc.evaluate(pred_rf)

print("=== Perbandingan Hasil ===")
print(f"Logistic Regression -> Accuracy: {acc_lr:.4f} | F1: {f1_lr:.4f} | AUC: {auc_lr:.4f}")
print(f"Random Forest       -> Accuracy: {acc_rf:.4f} | F1: {f1_rf:.4f} | AUC: {auc_rf:.4f}")

=== Perbandingan Hasil ===
Logistic Regression -> Accuracy: 0.7929 | F1: 0.7926 | AUC: 0.8650
Random Forest       -> Accuracy: 0.7929 | F1: 0.7927 | AUC: 0.8529


In [12]:
# D. Analisis Kesalahan Model
# Pilih model terbaik berdasarkan AUC
if auc_lr >= auc_rf:
    best_pred, nama_best = pred_lr, "Logistic Regression"
else:
    best_pred, nama_best = pred_rf, "Random Forest"
print("Model terbaik:", nama_best)

salah = best_pred.filter(col("ulasan_positif") != col("prediction"))

salah.select("nama_menu", "cabang", "waktu_pesan", "qty",
             "waktu_tunggu_menit", "ulasan_positif", "prediction").show(10)

print(f"Jumlah salah: {salah.count()} dari {best_pred.count()} data testing")

# Cari pola: kesalahan terjadi di waktu tunggu berapa?
salah.groupBy("waktu_tunggu_menit").count().orderBy("waktu_tunggu_menit").show(25)

Model terbaik: Logistic Regression
+---------+--------+-----------+---+------------------+--------------+----------+
|nama_menu|  cabang|waktu_pesan|qty|waktu_tunggu_menit|ulasan_positif|prediction|
+---------+--------+-----------+---+------------------+--------------+----------+
|Americano|Magelang|      Malam|  1|                15|             1|       0.0|
|Americano|Magelang|      Siang|  2|                22|             1|       0.0|
|Americano|Magelang|       Sore|  1|                 7|             0|       1.0|
|Americano|Magelang|       Sore|  2|                11|             0|       1.0|
|Americano|Magelang|       Sore|  3|                 5|             0|       1.0|
|Americano|Magelang|       Sore|  3|                13|             1|       0.0|
|Americano|Magelang|       Sore|  5|                 6|             0|       1.0|
|Americano|Magelang|       Sore|  5|                15|             1|       0.0|
|Americano|Semarang|      Malam|  1|                 2|        

# E. Kesimpulan Bisnis
Berdasarkan model Logistic Regression, faktor yang paling berpengaruh terhadap ulasan positif adalah waktu tunggu. Semakin lama pelanggan menunggu, semakin kecil peluang mereka memberi rating ≥ 4. Menu, cabang, waktu pesan, dan jumlah item pengaruhnya jauh lebih kecil. Analisis kesalahan juga menunjukkan model paling sering keliru di waktu tunggu sekitar 10–14 menit, yang berarti di rentang itulah kepuasan pelanggan paling mudah berubah. Rekomendasi untuk manajemen Kopi Nusantara: (1) tetapkan target waktu penyajian maksimal sekitar 10 menit di semua cabang; (2) beri perhatian khusus pada pesanan yang sudah menunggu mendekati 12 menit, misalnya didahulukan atau pelanggan diberi permintaan maaf atau bonus kecil; (3) catat dan pantau waktu tunggu tiap cabang secara rutin agar masalah antrean cepat ketahuan. Model ini akurasinya sekitar 79%, jadi sebaiknya dipakai sebagai alat bantu pemantauan, bukan satu-satunya dasar keputusan.

## Menutup SparkSession

In [13]:
spark.stop()
print("SparkSession ditutup.")

SparkSession ditutup.
